[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/02_%E7%9B%B8%E9%96%A2%E3%81%A8%E5%9B%9E%E5%B8%B0.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計3級-02. 相関と回帰直線

2つの量の関係を、散布図・相関係数・回帰直線で分析します。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')
df.head()

## 1. 散布図

2つの量的データの関係を点で表します。ここでは「数学」と「英語」の関係を見ます。

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df['数学'], df['英語'], alpha=0.6)
plt.xlabel('数学')
plt.ylabel('英語')
plt.title('数学と英語の関係')
plt.show()

## 2. 相関係数 r

関係の強さを −1〜+1 の数で表します。

| r の値 | 関係 |
|---|---|
| +0.7〜+1.0 | 強い正の相関 |
| +0.4〜+0.7 | やや正の相関 |
| −0.2〜+0.2 | ほぼ無相関 |
| −1.0〜−0.7 | 強い負の相関 |

In [ ]:
r = df['数学'].corr(df['英語'])
print('相関係数 r =', round(r, 3))

# 全教科の相関行列
df[['数学', '英語', '国語', '勉強時間']].corr().round(2)

> ⚠️ **相関 ≠ 因果**。「アイスの売上」と「水難事故」は相関するが、
原因は両方とも「気温（夏）」。見かけの相関に注意！

## 3. 回帰直線（最小二乗法）

点の真ん中を通る直線 `y = a·x + b` を引き、xからyを予測します。

In [ ]:
x = df['数学']
y = df['英語']
a, b = np.polyfit(x, y, 1)   # 傾きa・切片b
print(f'回帰直線: y = {a:.2f} x + {b:.2f}')

plt.figure(figsize=(6, 5))
plt.scatter(x, y, alpha=0.5)
xs = np.linspace(x.min(), x.max(), 100)
plt.plot(xs, a * xs + b, color='red', label=f'y={a:.2f}x+{b:.2f}')
plt.xlabel('数学'); plt.ylabel('英語'); plt.legend()
plt.show()

### 予測してみる
数学が85点の人の英語を予測すると？

In [ ]:
pred = a * 85 + b
print(f'数学85点 → 英語の予測 {pred:.1f}点')

---
## 🏆 練習問題

**問1.** `勉強時間` と `数学` の散布図を描き、相関係数を求めよう。正の相関はある？

**問2.** `勉強時間`(x) から `数学`(y) を予測する回帰直線を求め、
「3時間勉強した人」の数学点数を予測しよう。

**問3.** `国語` と `数学` の相関係数を求めよう。関係は強い？弱い？

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


<details><summary>解答例</summary>

```python
print(df['勉強時間'].corr(df['数学']))
a,b = np.polyfit(df['勉強時間'], df['数学'], 1)
print(a*3+b)
print(df['国語'].corr(df['数学']))
```
</details>